In [3]:
!pip install pandas
!pip install fredapi
!pip install yfinance

In [4]:
import pandas as pd

gold_data = pd.read_csv("Gold-Silver-GeopoliticalRisk_HistoricalData.csv")
gold_data['DATE'] = pd.to_datetime(gold_data['DATE'])
gold_data.set_index('DATE', inplace=True)
gold_data = gold_data[gold_data.index >= '1999-01-01']
gold_data


,GOLD_PRICE,GOLD_OPEN,GOLD_HIGH,GOLD_LOW,GOLD_CHANGE_%,SILVER_PRICE,SILVER_OPEN,SILVER_HIGH,SILVER_LOW,SILVER_CHANGE_%,GPRD,GPRD_ACT,GPRD_THREAT,EVENT
DATE,,,,,,,,,,,,,,
2025-09-10,3630.90,3633.61,3634.42,3620.90,-0.07,40.92,40.89,40.94,40.72,0.09,NaN,NaN,NaN,NaN
2025-09-09,3633.61,3637.10,3674.75,3625.33,-0.06,40.89,41.34,41.50,40.77,-1.13,NaN,NaN,NaN,NaN
2025-09-08,3635.84,3586.82,3646.60,3579.67,1.24,41.36,41.01,41.68,40.51,1.20,117.26,97.42,146.26,NaN
2025-09-07,3591.19,3592.07,3596.56,3586.95,0.12,40.86,41.00,41.01,40.76,-0.34,83.51,111.00,92.59,NaN
2025-09-05,3586.81,3547.00,3600.33,3540.05,1.15,41.01,40.69,41.44,40.55,0.76,166.42,110.61,224.05,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999-01-08,291.25,291.05,291.25,291.25,-0.10,5.29,5.22,5.29,5.28,0.76,70.81,61.38,92.16,NaN
1999-01-07,291.55,288.25,291.55,291.55,1.32,5.25,5.16,5.25,5.23,1.55,68.03,64.87,59.03,NaN
1999-01-06,287.75,287.00,287.75,287.75,0.28,5.17,5.03,5.17,5.15,2.58,62.19,71.79,43.55,NaN


In [5]:
from fredapi import Fred
fred = Fred(api_key="77607617679b9e388bc1c17f17bf0874")

end = '2025-09-10'
start = '1991-01-01'
df1 = pd.DataFrame({
'interest_rate' : fred.get_series('REAINTRATREARAT10Y', observation_start = start, observation_end = end),
'inflation_rate' : fred.get_series('FPCPITOTLZGUSA', observation_start = start, observation_end = end),
'expected_inflation' : fred.get_series('EXPINF10YR', observation_start = start, observation_end = end),
'gdp' : fred.get_series('GDP', observation_start = start, observation_end = end),
"us_m2_money_stock" : fred.get_series('WM2NS', observation_start = start, observation_end = end),
'wti_crude_oil_price' : fred.get_series('POILWTIUSDM', observation_start = start, observation_end = end)})

In [6]:
import yfinance as yf

# gold_futures = yf.download('MGC=F', start=start, end=end)
vix = yf.download('^VIX', start=start, end=end)
us_dollar = yf.download('DX-Y.NYB', start=start, end=end)
s_and_p_500 = yf.download('^GSPC', start=start, end=end)

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [7]:
raw_data = pd.concat([df1, s_and_p_500, vix, us_dollar, gold_data], axis=1)

raw_data.isna().sum()

raw_data.head()

,interest_rate,inflation_rate,expected_inflation,gdp,us_m2_money_stock,wti_crude_oil_price,"(Close, ^GSPC)","(High, ^GSPC)","(Low, ^GSPC)","(Open, ^GSPC)",...,GOLD_CHANGE_%,SILVER_PRICE,SILVER_OPEN,SILVER_HIGH,SILVER_LOW,SILVER_CHANGE_%,GPRD,GPRD_ACT,GPRD_THREAT,EVENT
1991-01-01,3.922931,4.234964,3.804597,6035.178,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-01-02,NaN,NaN,NaN,NaN,NaN,NaN,326.450012,330.750000,326.450012,330.200012,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-01-03,NaN,NaN,NaN,NaN,NaN,NaN,321.910004,326.529999,321.899994,326.459991,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-01-04,NaN,NaN,NaN,NaN,NaN,NaN,321.000000,322.350006,318.869995,321.910004,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1991-01-07,NaN,NaN,NaN,NaN,3309.9,NaN,315.440002,320.970001,315.440002,320.970001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Clean Up


## Model setup with target variable and redundant variable deletion

### Model 1: Gold price model
- **Target variable:** `GOLD_PRICE`. Other gold columns are removed because they describe the same asset directly, and silver columns are also removed so the model focuses on external drivers rather than another closely related metal price.
- **Market assets kept:** only **Close** is kept for the **S&P 500** (broad equity conditions), **VIX** (market stress), and **U.S. dollar index** (dollar strength). These give one interpretable price signal per asset and avoid repeated OHLC information.
- **Geopolitical variables kept/removed:** keep **`GPRD`** as the main geopolitical risk measure. Remove **`GPRD_ACT`** and **`GPRD_THREAT`** for the first model because they are closely related sub-indexes and may add overlapping information. Remove **`EVENT`** because it is sparse and mostly missing.



### Model 2: Silver catch-up model
- **Target variable:** `Future Silver Price (after 1 quarter) with new gold_silver_ratio(Gold/silver) as one of the predictors. This directly matches the idea that silver may catch up when the gold/silver ratio becomes high.
- **Market assets kept:** only **Close** is kept for the **S&P 500**, **VIX**, and **U.S. dollar index**, along with the macro variables. Keep **`GPRD`** as the main geopolitical control.
- **Columns removed:** repeated **Open/High/Low/Volume** fields are removed for each market asset, and raw current gold and silver columns are removed for now as they may be utilized later. **`GPRD_ACT`**, **`GPRD_THREAT`**, and **`EVENT`** are also removed for same reasons as model 1.


In [10]:
#Model 1: Only keep Gold_close out of gold variables, remove silver variables, then drop repeated Open/High/low, extra GPR variables
df = raw_data.copy()
df.columns = df.columns.map(str)

model1_df = df.drop(columns=[
    "('High', '^GSPC')","('Low', '^GSPC')","('Open', '^GSPC')","('Volume', '^GSPC')",
    "('High', '^VIX')","('Low', '^VIX')","('Open', '^VIX')","('Volume', '^VIX')",
    "('High', 'DX-Y.NYB')","('Low', 'DX-Y.NYB')","('Open', 'DX-Y.NYB')","('Volume', 'DX-Y.NYB')",
    "GOLD_OPEN","GOLD_HIGH","GOLD_LOW","GOLD_CHANGE_%",
    "SILVER_PRICE","SILVER_OPEN","SILVER_HIGH","SILVER_LOW","SILVER_CHANGE_%",
    "GPRD_ACT","GPRD_THREAT","EVENT"
], errors="ignore")

model1_df.head()

,interest_rate,inflation_rate,expected_inflation,gdp,us_m2_money_stock,wti_crude_oil_price,"('Close', '^GSPC')","('Close', '^VIX')","('Close', 'DX-Y.NYB')",GOLD_PRICE,GPRD
1991-01-01,3.922931,4.234964,3.804597,6035.178,NaN,NaN,NaN,NaN,83.070000,NaN,NaN
1991-01-02,NaN,NaN,NaN,NaN,NaN,NaN,326.450012,26.620001,82.790001,NaN,NaN
1991-01-03,NaN,NaN,NaN,NaN,NaN,NaN,321.910004,27.930000,82.809998,NaN,NaN
1991-01-04,NaN,NaN,NaN,NaN,NaN,NaN,321.000000,27.190001,83.360001,NaN,NaN
1991-01-07,NaN,NaN,NaN,NaN,3309.9,NaN,315.440002,28.950001,84.669998,NaN,NaN


In [11]:
# Model 2:future silver price target, ratio as predictor, then drop repeated Open/High/low, raw gold, extra GPR vars

df = raw_data.copy()
df.columns = df.columns.map(str)  # string cols
df["gold_silver_ratio"] = df["GOLD_PRICE"] / df["SILVER_PRICE"]  # predictor
df["future_silver_price_qtr"] = df["SILVER_PRICE"].shift(-63)  # ~1 trading quarter ahead

model2_df = df.drop(columns=[
    "(High, ^GSPC)","(Low, ^GSPC)","(Open, ^GSPC)","(Volume, ^GSPC)",
    "(High, ^VIX)","(Low, ^VIX)","(Open, ^VIX)","(Volume, ^VIX)",
    "(High, DX-Y.NYB)","(Low, DX-Y.NYB)","(Open, DX-Y.NYB)","(Volume, DX-Y.NYB)",
    "GOLD_OPEN","GOLD_HIGH","GOLD_LOW","GOLD_CHANGE_%", "GOLD_PRICE",	"SILVER_PRICE",
    "SILVER_OPEN","SILVER_HIGH","SILVER_LOW","SILVER_CHANGE_%",
    "GPRD_ACT","GPRD_THREAT","EVENT"
], errors="ignore")
model2_df.head()


,interest_rate,inflation_rate,expected_inflation,gdp,us_m2_money_stock,wti_crude_oil_price,"('Close', '^GSPC')","('High', '^GSPC')","('Low', '^GSPC')","('Open', '^GSPC')",...,"('Open', '^VIX')","('Volume', '^VIX')","('Close', 'DX-Y.NYB')","('High', 'DX-Y.NYB')","('Low', 'DX-Y.NYB')","('Open', 'DX-Y.NYB')","('Volume', 'DX-Y.NYB')",GPRD,gold_silver_ratio,future_silver_price_qtr
1991-01-01,3.922931,4.234964,3.804597,6035.178,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,83.070000,83.430000,83.010002,83.370003,0.0,NaN,NaN,NaN
1991-01-02,NaN,NaN,NaN,NaN,NaN,NaN,326.450012,330.750000,326.450012,330.200012,...,26.620001,0.0,82.790001,83.080002,82.660004,82.930000,0.0,NaN,NaN,NaN
1991-01-03,NaN,NaN,NaN,NaN,NaN,NaN,321.910004,326.529999,321.899994,326.459991,...,27.930000,0.0,82.809998,82.930000,82.680000,82.889999,0.0,NaN,NaN,NaN
1991-01-04,NaN,NaN,NaN,NaN,NaN,NaN,321.000000,322.350006,318.869995,321.910004,...,27.190001,0.0,83.360001,83.519997,82.599998,82.669998,0.0,NaN,NaN,NaN
1991-01-07,NaN,NaN,NaN,NaN,3309.9,NaN,315.440002,320.970001,315.440002,320.970001,...,28.950001,0.0,84.669998,85.110001,84.540001,84.870003,0.0,NaN,NaN,NaN


## Removing NAs

In [12]:
model1_df = model1_df.dropna(subset=["GOLD_PRICE"])  # drop rows with no gold target
x1_cols = model1_df.columns.drop("GOLD_PRICE")  # non-target cols
model1_df[x1_cols] = model1_df[x1_cols].ffill()  # carry last value forward in all feature variables if NA
model1_df = model1_df.dropna()  # drop the few leading rows still NA

model1_df.head()



,interest_rate,inflation_rate,expected_inflation,gdp,us_m2_money_stock,wti_crude_oil_price,"('Close', '^GSPC')","('Close', '^VIX')","('Close', 'DX-Y.NYB')",GOLD_PRICE,GPRD
2008-01-01,1.399491,3.8391,2.174666,14706.538,7526.4,92.982609,1468.359985,22.500000,76.699997,833.7,63.37
2008-01-02,1.399491,3.8391,2.174666,14706.538,7526.4,92.982609,1447.160034,23.170000,75.970001,857.2,95.00
2008-01-03,1.399491,3.8391,2.174666,14706.538,7526.4,92.982609,1447.160034,22.490000,75.889999,863.4,82.81
2008-01-04,1.399491,3.8391,2.174666,14706.538,7526.4,92.982609,1411.630005,23.940001,75.790001,861.5,87.35
2008-01-07,1.399491,3.8391,2.174666,14706.538,7551.9,92.982609,1416.180054,23.790001,76.169998,857.9,78.18


In [24]:

x2_cols = model2_df.columns.drop("future_silver_price_qtr")  # features only
model2_df[x2_cols] = model2_df[x2_cols].ffill()  # carry last value forward
model2_df = model2_df.dropna()  # remove remaining NAs
model2_df.head()

,interest_rate,inflation_rate,expected_inflation,gdp,us_m2_money_stock,wti_crude_oil_price,"('Close', '^GSPC')","('High', '^GSPC')","('Low', '^GSPC')","('Open', '^GSPC')",...,"('Open', '^VIX')","('Volume', '^VIX')","('Close', 'DX-Y.NYB')","('High', 'DX-Y.NYB')","('Low', 'DX-Y.NYB')","('Open', 'DX-Y.NYB')","('Volume', 'DX-Y.NYB')",GPRD,gold_silver_ratio,future_silver_price_qtr
1999-01-04,2.308152,2.188027,2.587127,9411.682,4439.9,12.435714,1228.099976,1248.810059,1219.099976,1229.229980,...,25.379999,0.0,93.440002,94.150002,93.050003,93.889999,0.0,54.95,58.168016,5.04
1999-01-05,2.308152,2.188027,2.587127,9411.682,4439.9,12.435714,1244.780029,1246.109985,1228.099976,1228.099976,...,25.920000,0.0,93.470001,93.820000,93.080002,93.440002,0.0,60.68,56.934524,5.16
1999-01-06,2.308152,2.188027,2.587127,9411.682,4439.9,12.435714,1272.339966,1272.500000,1244.780029,1244.780029,...,23.360001,0.0,94.529999,94.849998,93.209999,93.739998,0.0,62.19,55.657640,4.93
1999-01-07,2.308152,2.188027,2.587127,9411.682,4439.9,12.435714,1269.729980,1272.339966,1257.680054,1272.339966,...,24.420000,0.0,93.720001,94.480003,93.449997,94.419998,0.0,68.03,55.533333,4.97
1999-01-08,2.308152,2.188027,2.587127,9411.682,4439.9,12.435714,1275.089966,1278.239990,1261.819946,1269.729980,...,22.950001,0.0,94.349998,94.820000,93.660004,93.800003,0.0,70.81,55.056711,4.97


## Variable Description
This table lists the main working variables used across both models covering **1991-01-01 to 2025-09-09**.

| Variable (units) | What it measures |
|---|---|
| `interest_rate` (%) | U.S. 10-year real interest rate. |
| `inflation_rate` (annual %) | U.S. CPI inflation rate. |
| `expected_inflation` (%) | U.S. expected average inflation over the next 10 years. |
| `gdp` (billions of USD, SAAR) | U.S. nominal gross domestic product. |
| `us_m2_money_stock` (billions of USD) | U.S. M2 money supply / broad household liquidity. |
| `wti_crude_oil_price` (USD/barrel) | Global WTI crude oil price. |
| `(Close, ^GSPC)` (index level) | S&P 500 closing level, measuring broad U.S. stock market performance. |
| `(Close, ^VIX)` (index level) | VIX closing level, measuring expected near-term U.S. stock market volatility. |
| `(Close, DX-Y.NYB)` (index level) | U.S. Dollar Index closing level, measuring U.S. dollar strength against a basket of major currencies. |
| `GPRD` (index, 1985–2019 = 100) | Daily geopolitical risk index. |
| `GOLD_PRICE` (USD-based gold price series) | Daily gold price level. |
| `SILVER_PRICE` (USD-based silver price series) | Daily silver price level. |
| `gold_silver_ratio` (unitless ratio) | Gold price divided by silver price. |
| `future_silver_price_qtr` (same units as `SILVER_PRICE`) | Silver price about one trading quarter later. |